> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# Can I trust this data? Every number in the video

Each cell reproduces one claim from the video from a seeded simulation (the same functions the animations use). Change the inputs and see how the conclusions move. Requirements: `numpy`.

In [ ]:
import numpy as np

SQ = np.sqrt(252)

def sharpe(x):
    return float(x.mean() / x.std() * SQ)

def pct(x, digits=1):
    v = round(x * 100, digits) + 0.0                                      # never print "-0.0"
    return f"{v:+.{digits}f}%"

## 1. Survivorship: a backtest of today's list
200 made-up companies, 10 years of monthly prices, 30% volatility, **zero expected return**. A company leaves the list once it falls to half its starting price (a leaver is sold at that price). Compare buying today's survivors with buying everyone.

In [ ]:
def survivors(seed=3, n=200, years=10, vol=0.30, floor=0.5):
    """Made-up companies with zero expected return. A company leaves the list once it falls to `floor`."""
    m = 12 * years
    s = vol / np.sqrt(12)
    p = np.exp(np.cumsum(np.random.default_rng(seed).normal(-s * s / 2, s, (n, m)), axis=1))
    p = np.hstack([np.ones((n, 1)), p])                                   # month 0 = 1.0
    below = p < floor
    out = below.any(axis=1)
    exit_m = np.where(out, below.argmax(axis=1), m)
    held = p.copy()                                                       # a leaver is sold at its exit price
    for i in np.where(out)[0]:
        held[i, exit_m[i]:] = p[i, exit_m[i]]
    surv_avg = p[~out].mean(axis=0)
    all_avg = held.mean(axis=0)
    cagr = lambda v: v ** (1 / years) - 1                                 # noqa: E731
    return p, out, exit_m, surv_avg, all_avg, cagr(surv_avg[-1]), cagr(all_avg[-1])

p, out, exit_m, surv_avg, all_avg, g_surv, g_all = survivors()
print(f"left the list: {out.sum()} of {len(out)}")
print(f"survivors only: {pct(g_surv)} a year   everyone: {pct(g_all)} a year")

## 2. Filled gaps break correlation
Two markets with a true daily correlation of 0.6. In each, one day in ten (different days) is filled with the last price; the next real day catches up.

In [ ]:
def stale_fill(r, mask):
    """Fill masked days with the last price (return 0); the next real day catches up."""
    out, acc = r.copy(), 0.0
    for t in range(len(r)):
        if mask[t]:
            acc += r[t]
            out[t] = 0.0
        else:
            out[t] = r[t] + acc
            acc = 0.0
    return out

def two_markets(seed=21, n=2520, rho=0.6, frac=0.10):
    rng = np.random.default_rng(seed)
    z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n) * 0.01
    ma, mb = rng.random(n) < frac, rng.random(n) < frac
    a, b = stale_fill(z[:, 0], ma), stale_fill(z[:, 1], mb)
    return z, a, b, ma, mb, float(np.corrcoef(z.T)[0, 1]), float(np.corrcoef(a, b)[0, 1])

z, a, b, ma, mb, rho_true, rho_fill = two_markets()
print(f"true correlation {rho_true:.2f}   measured after filling {rho_fill:.2f}")
print(f"daily volatility: true {z[:, 0].std():.4%}   filled {a.std():.4%}  (barely moves)")

## 3. An unadjusted split
A 2-for-1 split on day 150. The unadjusted file shows a crash that never happened.

In [ ]:
def split_path(seed=8, n=250, day=150):
    r = np.random.default_rng(seed).normal(0, 0.015, n)
    real = 200 * np.exp(np.cumsum(r))
    raw = real.copy()
    raw[day:] /= 2                                                        # what an unadjusted file shows
    adj = real / 2                                                        # history divided by the split ratio
    stats = {}
    for name, series in (("adj", adj), ("raw", raw)):
        lr = np.diff(np.log(series))
        stats[name] = (float(np.exp(lr.min()) - 1), float(lr.std() * SQ))
    return real, raw, adj, stats

real, raw, adj, stats = split_path()
for name in ('adj', 'raw'):
    worst, vol = stats[name]
    print(f"{name:4s} worst day {worst:+.1%}   annual volatility {vol:.0%}")

## 4. Futures: glued prices vs a rolled position
Quarterly contracts priced at spot × (1 + carry × quarters to expiry); the next contract is 1.5% cheaper (backwardation).

In [ ]:
def futures(seed=1, years=10, carry=-0.015, q=63):
    """Contracts priced at spot x (1 + carry x quarters to expiry); the next one is 1.5% cheaper (backwardation)."""
    days = q * 4 * years
    spot = 40 * np.exp(np.cumsum(np.random.default_rng(seed).normal(0, 0.012, days)))
    tte = q - (np.arange(days) % q)
    front = spot * (1 + carry * tte / q)
    nxt = spot * (1 + carry * (tte + q) / q)
    roll = np.where(np.arange(1, days) % q == 0)[0]                       # last day of each contract
    glued_r = np.diff(np.log(front))
    honest_r = glued_r.copy()
    for i in roll:
        honest_r[i] = np.log(front[i + 1] / nxt[i])                      # the new contract's own move
    glued = 100 * front / front[0]
    rolled = 100 * np.exp(np.r_[0, np.cumsum(honest_r)])
    back = front.copy()
    for i in roll[::-1]:
        back[: i + 1] += nxt[i] - front[i]                                # shift older history by each roll gap
    return spot, front, nxt, roll, glued, rolled, back

spot, front, nxt, roll, glued, rolled, back = futures()
print(f"glued prices: {glued[-1] / 100 - 1:+.0%}   holding and rolling: {rolled[-1] / 100 - 1:+.0%}")
print(f"back-adjusted history starts at {back[0]:.1f}; the real price was {front[0]:.1f}; lowest back-adjusted level {back.min():.1f}")

## 5. Leakage: tomorrow inside a moving average
"Buy when below its 10-day average" on pure noise. One version's average accidentally includes tomorrow's price.

In [ ]:
def leak_demo(seed=31, n=2520, L=10):
    """'Buy when below its L-day average' on pure noise, with and without tomorrow inside the average."""
    r = np.random.default_rng(seed).normal(0, 0.01, n)
    p = 100 * np.exp(np.cumsum(r))

    def avg(lead):
        out = np.full(n, np.nan)
        for t in range(L, n - 1):
            out[t] = p[t - L + 1 + lead:t + 1 + lead].mean()
        return out

    def run(m, delay=0):
        pos = np.roll(np.sign(m - p), delay)
        pnl = pos[:-1] * r[1:]
        ok = ~np.isnan(pnl)
        ok[:L + delay + 1] = False
        return sharpe(pnl[ok])

    leak, honest = avg(1), avg(0)
    return p, r, run(leak), run(honest), run(leak, 1)

p, r, s_leak, s_honest, s_leak_late = leak_demo()
print(f"Sharpe with tomorrow inside: {s_leak:.2f}   honest: {s_honest:+.2f}")

## 6. Check 1: shift a day
Delay every input by one more day. A leak collapses; a real edge built on yesterday's information barely notices.

In [ ]:
def planted(seed=33, n=2520, L=100):
    """A slow trend planted in noise, and a 100-day trend rule, with and without one extra day of delay."""
    rng = np.random.default_rng(seed)
    mu = np.zeros(n)
    for t in range(1, n):
        mu[t] = 0.999 * mu[t - 1] + rng.normal(0, 0.00004)
    r = mu + rng.normal(0, 0.01, n)
    p = 100 * np.exp(np.cumsum(r))
    m = np.convolve(p, np.ones(L) / L, "full")[:n]

    def run(delay):
        pos = np.roll(np.sign(p - m), delay)
        pos[:L + delay] = 0
        return sharpe((pos[:-1] * r[1:])[L + 1:])

    return run(0), run(1)

s_edge, s_edge_late = planted()
print(f"leaky rule: {s_leak:+.2f}  ->  {s_leak_late:+.2f} with one extra day")
print(f"planted edge: {s_edge:+.2f}  ->  {s_edge_late:+.2f} with one extra day")

## 7. Check 2: plant something
Over 10 years, t ≈ Sharpe × √years. The pipeline should find the planted edge, find nothing in noise, and a 'strong signal' in noise means a bug.

In [ ]:
for name, s in (("planted edge", s_edge), ("pure noise, honest rule", s_honest), ("pure noise, leaky code", s_leak)):
    print(f"{name:24s} t = {s * np.sqrt(10):+.1f}")

### Try this
- Survivorship: lower the exit level to 0.3 (a 70% fall). How much of the survivors' edge is left?
- Filled gaps: fill one day in five. How far does the correlation fall?
- Leakage: make the average 50 days long. Does the leak get quieter or louder?

*Simulated results on synthetic data. Education only. Not financial advice.*